In [10]:
from pathlib import Path
from nuscenes.nuscenes import NuScenes
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Current dir
DATA_DIR = Path.home() / "Desktop" / "gnn_trajectory" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {DATA_DIR}")
VERSION = "v1.0-mini"

Data directory: /home/danilo/Desktop/gnn_trajectory/data


Download NuScenes mini-dataset: [nuScenes](https://www.nuscenes.org/) 

In [11]:
# Download dataset 
!wget -P {DATA_DIR} https://www.nuscenes.org/data/v1.0-mini.tgz
!tar -xzf {DATA_DIR}/v1.0-mini.tgz -C {DATA_DIR}
!ls {DATA_DIR}

--2026-09-17 01:22:51--  https://www.nuscenes.org/data/v1.0-mini.tgz
Resolving www.nuscenes.org (www.nuscenes.org)... 3.167.2.76, 3.167.2.35, 3.167.2.38, ...
Connecting to www.nuscenes.org (www.nuscenes.org)|3.167.2.76|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4167696325 (3.9G) [application/x-tar]
Saving to: ‘/home/danilo/Desktop/gnn_trajectory/data/v1.0-mini.tgz’

v1.0-mini.tgz       100%[===================>]   3.88G  12.2MB/s    in 2m 46s  

2026-09-17 01:25:36 (24.0 MB/s) - ‘/home/danilo/Desktop/gnn_trajectory/data/v1.0-mini.tgz’ saved [4167696325/4167696325]

maps  samples  sweeps  v1.0-mini  v1.0-mini.tgz


In [ ]:
# Load dataset
nusc = NuScenes(
    version=VERSION,
    dataroot=str(DATA_DIR),
    verbose=True
)

In [ ]:
# List of scenes
print("NºScenes:", len(nusc.scene))
print("NºSamples:", len(nusc.sample))
print("Annotations:", len(nusc.sample_annotation)) # annotations are the objects in the scene
print("Instances:", len(nusc.instance)) # instances are the unique objects in the scene
print("Categories:", len(nusc.category)) # categories are the types of objects in the scene

# List of categories
categories = [cat['name'] for cat in nusc.category]
print(f"Categories:", categories)

NºScenes: 10
NºSamples: 404
Annotations: 18538
Instances: 911
Categories: 23
Categories: ['human.pedestrian.adult', 'human.pedestrian.child', 'human.pedestrian.wheelchair', 'human.pedestrian.stroller', 'human.pedestrian.personal_mobility', 'human.pedestrian.police_officer', 'human.pedestrian.construction_worker', 'animal', 'vehicle.car', 'vehicle.motorcycle', 'vehicle.bicycle', 'vehicle.bus.bendy', 'vehicle.bus.rigid', 'vehicle.truck', 'vehicle.construction', 'vehicle.emergency.ambulance', 'vehicle.emergency.police', 'vehicle.trailer', 'movable_object.barrier', 'movable_object.trafficcone', 'movable_object.pushable_pullable', 'movable_object.debris', 'static_object.bicycle_rack']


Inspect one sample in a single scene and its annotations 

In [ ]:
scene = nusc.scene[0]
print((scene)) # it's a dict
print("Scene description:", scene['description'])

# Get first sample
sample = nusc.get("sample", scene['first_sample_token'])
print("Sample keys:", sample.keys()) # keys

{'token': 'cc8c0bf57f984915a77078b10eb33198', 'log_token': '7e25a2c8ea1f41c5b0da1e69ecfa71a2', 'nbr_samples': 39, 'first_sample_token': 'ca9a282c9e77460f8360f564131a8af5', 'last_sample_token': 'ed5fc18c31904f96a8f0dbb99ff069c0', 'name': 'scene-0061', 'description': 'Parked truck, construction, intersection, turn left, following a van'}
Scene description: Parked truck, construction, intersection, turn left, following a van
Sample keys: dict_keys(['token', 'timestamp', 'prev', 'next', 'scene_token', 'data', 'anns'])


In [ ]:
# Explore keys 
print(f"Timestamp:", sample['timestamp'])
print("Number of annotations:", len(sample["anns"]))
print()

for ann_token in sample["anns"]:
    ann = nusc.get("sample_annotation", ann_token)
    print(ann["category_name"])
    
target_ann = None 
for ann_token in sample["anns"]: 
    ann = nusc.get("sample_annotation", ann_token) # get the token of the annotation
    
    # find one normal car in the scene (note that the scene may have many cars) 
    if ann["category_name"] == "vehicle.car":
        target_ann = ann
        break

Timestamp: 1532402927647951
Number of annotations: 69

human.pedestrian.adult
human.pedestrian.adult
vehicle.car
human.pedestrian.adult
movable_object.trafficcone
vehicle.bicycle
human.pedestrian.adult
vehicle.car
human.pedestrian.adult
movable_object.barrier
movable_object.barrier
human.pedestrian.adult
human.pedestrian.adult
human.pedestrian.adult
human.pedestrian.adult
movable_object.barrier
vehicle.car
human.pedestrian.adult
vehicle.truck
vehicle.car
human.pedestrian.adult
movable_object.barrier
movable_object.barrier
movable_object.barrier
movable_object.trafficcone
movable_object.barrier
vehicle.bus.rigid
human.pedestrian.adult
human.pedestrian.adult
movable_object.barrier
human.pedestrian.adult
human.pedestrian.adult
movable_object.barrier
human.pedestrian.adult
human.pedestrian.adult
movable_object.barrier
vehicle.car
movable_object.barrier
movable_object.barrier
human.pedestrian.adult
vehicle.car
movable_object.barrier
movable_object.barrier
vehicle.construction
movable_object

In [ ]:
target_ann # contains information for a annotation (i.e a car) in this specific Timestamp

{'token': '924ee6ac1fed440a9d9e3720aac635a0',
 'sample_token': 'ca9a282c9e77460f8360f564131a8af5',
 'instance_token': 'bd26c2cdb22d4bb1834e808c89128898',
 'visibility_token': '3',
 'attribute_tokens': ['c3246a1e22a14fcb878aa61e69ae3329'],
 'translation': [353.794, 1132.355, 0.602],
 'size': [2.011, 4.633, 1.573],
 'rotation': [0.9797276292877292, 0.0, 0.0, -0.20033415188191459],
 'prev': '',
 'next': 'f0cbd9dbafd74e20bcf6dd0357c97f59',
 'num_lidar_pts': 5,
 'num_radar_pts': 0,
 'category_name': 'vehicle.car'}

$
\\ \text{scene} \\ \downarrow \\ \text{sample = one timestamp} \\ \downarrow \\ \text{sample annotation = one object at that timestamp} \\ \downarrow \\ \text{instance token = identity of that same physical object across time}
$

Follow the target vehicle through time

In [ ]:
positions = []
timestamps = []

ann = target_ann

# Go to first annotation of this instance
while ann["prev"] != :

SyntaxError: incomplete input (569501263.py, line 7)